# 见微知著 - Decoding EEG Movement Imagination

Codebook by Antonia Reul | Contact: areul@uni-osnabrueck.de

**Main research question:** 

    Can a hybrid CNN-Transformer architecture reliably predict imagined, but not executed movements based on EEG recordings?


This Jupyter notebook combines the five main .py files, was developed to test the pipeline all in one place and display plots directly, so that interested readers can understand and work with the code easily.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mne 
import math
from mne.io import concatenate_raws, read_raw_edf
from mne.datasets import eegbci
from mne.preprocessing import ICA
from sklearn.model_selection import train_test_split
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

/opt/miniconda3/envs/dl26/lib/python3.12/site-packages/mne/externals/tempita/__init__.py:35: DeprecationWarning: 'cgi' is deprecated and slated for removal in Python 3.13
  import cgi


## Dataset.py
_Importing and preprocessing the data of each subject_

The Physionet dataset contains EEG data recorded for different task categories. Since the focus of this project is on movement imagery, the runs for left vs. right fist imagination (runs=[4, 8, 12]) have been selected. Using the runs for both fist vs. both feet imagination could have also been possible or comparing movement imagination vs. movement execution data, although that would alter the research question of this project. Data has been selected from 105 subjects. The complete dataset contains 109 subjects, but four subjects have repeatedly been reported to be unsuitable for working with (more below). Each subject performed about 15 task trials per run, so approximately 45 trials in total (3 runs). This leads to data with about 4700 trials of shape (64, 481) and with a 80/20 test-train split, about 3800 training trials. 

#### Data Exclusion
Subjects 88, 89, 92 and 100 have been excluded since subject 89 had incorrect labels and the sampling rate of the other three subjects was recorded at 128 Hz instead of the original 160 Hz (Roots et al., 2020; Köllőd et al., 2023). 

In [ ]:
"""A plotted analysis of the dataset can be found in a 
separate Jupyter notebook in the folder 'notebooks'."""

# Dataset documentation: https://www.physionet.org/content/eegmmidb/1.0.0/
# MNE: https://mne.tools/stable/generated/mne.datasets.eegbci.load_data.html

class PreprocessedDataset(Dataset):
    """
    Initialize the dataset loader for the eegbci dataset
    
    Parameters:
    - subject_ids: List of subject IDs to load
    - runs: List of run numbers (e. g. [4] for left vs. right hand)
    - preload: Whether to load data into memory
    - filter_freqs: Tuple of low and high frequency for bandpass filtering
    - baseline: Baseline correction (e. g. "prestim" or None)
    """
    
    def __init__(self, subject_ids=None, runs=None, preload=True, baseline=(-0.5, 0)):
        BAD = {88, 89, 92, 100}
        self.subject_ids = subject_ids if subject_ids else [s for s in range(1, 110) if s not in BAD] 
                            # all subjects with reliable data
        self.runs = runs if runs else [4, 8, 12] # runs for imagined left/right fist
        self.preload = preload
        self.baseline = baseline
        self.X = None 
        self.y = None

    def load_subject_data(self, subject_id: int) -> Tuple[np.ndarray, np.ndarray]:
        # Loading the raw data file for a single subject
        paths = eegbci.load_data(subject_id, runs=self.runs, preload=self.preload, update_path=True)
        raw = concatenate_raws([read_raw_edf(p, preload=True) for p in paths])
        events, event_id = mne.events_from_annotations(raw, event_id=dict(T0=1, T1=2, T2=3))

        eegbci.standardize(raw)    
        # 10-10 system used excluding Nz, F9/F10, ...
        raw.set_montage(mne.channels.make_standard_montage('standard_1005'))

        # Apply notch and high-pass filter (2nd needed for ICA)
        raw.notch_filter(freqs=[60, 120])
        raw_for_ica = raw.copy().filter(l_freq=1.0, h_freq=None)

        # Apply ICA to filtered copy 
        ica = ICA(n_components=0.99, random_state=42, method='fastica')
        ica.fit(raw_for_ica)

        # Find and apply components to original raw data
        # using frontal-polar electrodes closest to eyes 
        # -> most sensitive to EOG signals
        eog_ind = ica.find_bands_eog(raw, ch_name=['Fp1', 'Fp2'])
        ica.exclude = eog_ind 
        ica.apply(raw)

        # Define event ID
        event_id = {"left": 2, "right": 3}

        # Epoching into 4 s windows
        epochs = mne.Epochs(raw, events=events, event_id=event_id, 
                                tmin=-0.5, tmax=4.0, baseline=self.baseline, 
                                preload=self.preload, reject=dict(eeg=150e-6), 
                                flat=dict(eeg=1e-7))

        # Extract data and labels
        X = epochs.get_data().astype(np.float32) * 1e6 # conversion to µV
         # (n_epochs, n_channels, n_samples)
        
        # Normalize data by z-score normalization 
        # Calculate mean and std across epoch and time dim per channel
        mu = np.mean(X, axis=(0, 2), keepdims=True)
        sd = np.std(X, axis=(0, 2), keepdims=True)
        X = (X - mu) / (sd + 1e-8)

        y = epochs.events[:, 2] # Event IDs from 3rd column (2 for left, 3 for right)

        # Raw event IDs do not start at 0 which PyTorch classification losses expect 
        # -> map to (0, 1) binary scale
        label_map = {2: 0, 3: 1}

        # Target vector y
        # T1 = 0 and T2 = 1
        y = np.array([label_map[label] for label in y])

        return X, y

    def __len__(self):
        # Return total number of samples
        return len(self.X)

    def __getitem__(self, idx):
        # Access a single sample by index
        return self.X[idx], self.y[idx]

    def load_data(self) -> None:
        # Load and process data for all subjects
        self.X_subjects = []
        self.y_subjects = []

        for subject_id in self.subject_ids:
            X, y = self.load_subject_data(subject_id)
            self.X_subjects.append(X)
            self.y_subjects.append(y)
        
        # Concatenate all data from subjects 
        self.X = np.concatenate(self.X_subjects, axis=0)
        self.y = np.concatenate(self.y_subjects, axis=0)

        return self.X, self.y 
        # (bath, channels, samples)

## Datamodule.py
_creating data loaders to organize the data into small batches_

In [ ]:
def create_dataloaders(X, y):
    """
    Create data loaders to organize the data into small batches
    """

    BAD = {88, 89, 92, 100}
    all_subjects = list(s for s in range(1, 110) if s not in BAD)

    # 80 % of subjects used for training, 20 % for testing
    train_subjects, test_subjects = train_test_split(all_subjects, test_size=0.2, random_state=42)

    train_dataset = PreprocessedDataset(subject_ids=train_subjects)
    test_dataset = PreprocessedDataset(subject_ids=test_subjects)

    train_dataset.load_data()
    test_dataset.load_data()
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True) 
    # Some other projects use batch size of 16 with the PhysioNet dataset
    test_loader = DataLoader(test_dataset, batch_size=32)

    return train_loader, test_loader

## Model.py
_Different classes for both the main and comparison models_

MLPs are unsuitable for working with EEG data. The hidden layers require lots of parameters (EEG data contains many features), so the MLP is very likely to just overfit (memorize noise of) the EEG data, which has a low signal-to-noise ration, rather than actually learn meaningful representations. To work with the EEG data, it would have to get flattened, which effectively destroys spatial and temporal information by just collapsing them onto one dimension. Instead, global average pooling (GAP) can also be used. To reduce the dimensionality of the vector, GAP takes the average of the time dimension for each channel. Although it may appear counterintuitive, the MLP (name: PoorMLP) is used here as a sanity check in comparison to the CNN-Transformer. It is expected to definitely worse in the classification task due to the reasons stated above. In order to still have a reasonable baseline, a one layer CNN is used as the main baseline model. Users testing this code with fragile laptops may also be advised to outcomment the use of the PoorMLP model - with a couple of million parameters approximately, it can be quite computationally expensive. Since this is a university project and I have worked with it, I decided to leave it in here, nevertheless, but not focus on it too much during analysis.

In [ ]:
class PoorMLP(nn.Module):
    """
    A small and simple baseline model to compare the main model to
    
    Parameters:
    - in_dim: Dimension of the input data
    - num_classes: Number of classes (2)
    - hidden_dim: Hidden dimension size of MLP
    - dropout_rate: Dropout rate
    """

    def __init__(self, in_dim, num_classes=2, hidden_dim=128, dropout_rate=0.5) -> None:
        # EEG datasets small & noisy -> strong dropout suggested
        # 64 channels/electrodes -> 1:1 mapping
        super(PoorMLP, self).__init__()
        
        self.layer_1 = nn.Linear(in_dim, hidden_dim)
        self.layer_2 = nn.Linear(hidden_dim, hidden_dim // 2)
        # Enforce dense representations (hidden_dim // 2)
        self.layer_out = nn.Linear(hidden_dim // 2, num_classes)
        self.dropout = nn.Dropout(dropout_rate)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = torch.mean(x, dim=2) # (batch, channels)
        x = self.relu(self.layer_1(x))
        x = self.dropout(x)
        x = self.relu(self.layer_2(x))
        x = self.dropout(x)
        x = self.layer_out(x)
        return x

Instead of using a MLP for the classification task, a simple CNN can be used. Using convolutional and pooling layers, CNNs act as efficient feature extractors. However due to their local inductive bias, they struggle with capturing global relationships, which is why research has shifted to focusing on hybrid architectures in recent years, which will be described further below.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(BaselineCNN, self).__init__()

        self.conv1 = nn.Conv1d(in_dim=64, out_dim=16x, kernel_size=25)
        self.pool = nn.AdaptiveAvgPool1d(1) # Reduce time dimension to 1
        self.fc = nn.Linear(16, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x)) # (batch, 16, samples-24)
        x = self.pool(x).squeeze(-1) # (batch, 16)
        x = self.fc(x)               # (batch, 2)
        return x

When analysing EEG data, temporal (frequency) should occur before spatial filtering, i. e. detecting which electrodes are important. If 1D convolutional layers are used, instead of 2D, the information from all electrodes get mixed up. Instead, the goal should be to analyse the time dimension for each electrode independently, using a temporal filter first, as has been done in popular networks like EEGNet (Lawhern et al., 2018). Further, ELU is used instead of ReLU since it allows for negative values which is useful when dealing with zero-mean oscillatory signals. After every convolution, batch normalization is used to ensure stability since EEG signals are noisy and highly variable between subjects. Only 16 temporal filters (F1) are used since they have been shown to generally be sufficient to cover primary frequency ranges of interest in MI, as has been shown in the EEGNet and ShallowConvNet papers. Using too many filters, like 64, may lead to the model drastically overfitting. Besides that, for every temporal filter D spatial filters are created, so that the model can look for different spatial topographies for the same frequency. Moreover, the pool size for the global average pooling has been chosen since it is the standard balance used in the EEGNet and ShallowConvNet literature for sampling rates between 128 Hz and 256 Hz. One could also experiment with the values 8 and 2 instead, to compare how model performance improves or worsens.

In [ ]:
class CNN(nn.Module):
    """
    CNN to capture local relationships before input is passed to transformer

    Parameters:
    - n_channels: Number of channels, 64 in our dataset
    - num_classes: Number of classes, 2
    - emb_dim: Dimension of embeddings
    - fs: Sampling rate
    """

    def __init__(self, n_channels=64, num_classes=2, emb_dim=128, fs=160):
        super(CNN, self).__init__()

        F1 = 16 # Number of temporal filters
        D = 2 # Depth multiplier for spatial filters
        k_t = fs // 10 # Temporal kernel size, here: 16

        # Temporal Convolution: Learn frequency/band-pass filters
        # input: (batch, 1, n_channels, time)
        self.temp_conv = nn.Conv2d(1, F1, (1, k_t), padding=(0, k_t // 2), bias=False)
        self.bn1 = nn.BatchNorm2d(F1)

        # Spatial Convolution: Learn spatial filters
        self.spat_conv = nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False)
        self.bn2 = nn.BatchNorm2d(F1 * D)

        self.elu = nn.ELU()
        self.pool = nn.AvgPool2d(kernel_size=(1, 4)) # Pool along time dimension
        self.embedding_layer = nn.Linear(F1 * D, emb_dim)

    def forward(self, x):
        # x input shape: (batch, n_channels, time)
        # Reshape to (batch, 1, n_channels, time) for Conv2d
        x = x.unsqueeze(1)

        x = self.temp_conv(x)
        x = self.bn1(x)
        x = self.elu(x)

        x = self.spat_conv(x)
        x = self.bn2(x)
        x = self.elu(x)

        x = self.pool(x) # (batch, F1*D, 1, time_reduced)

        # Transform to 3D for transformer
        x = x.squeeze(2) # (batch, F1*D, time_reduced)
        x = x.transpose(1, 2) # (batch, time_reduced, F1*D)

        x = self.embedding_layer(x) # (batch, time_reduced, emb_dim)

        return x

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Positional encoding so important temporal information 
    is not lost when input is passed onto Transformer

    Parameters:
    - emb_dim: Dimension of embeddings
    - max_patches: Maximum number of patches
    """
    
    def __init__(self, emb_dim, max_patches):
        super(PositionalEncoding, self).__init__()
        self.emb_dim = emb_dim
        assert emb_dim % 2 == 0 # must be even for sin/cos pairs

        # Positional encoding matrix
        pe = torch.zeros(max_patches, emb_dim)

        # Sinusoidal positional encoding
        # Position indices
        pos = torch.arange(0, max_patches, dtype=torch.float).unsqueeze(1) # (max_patches, 1)

        # Division term (tensor of even indices since pose alternate between sin/cos)
        div_term = torch.exp(torch.arange(0, emb_dim, 2).float() * (-math.log(10000.0) / emb_dim))

        pe[:, 0::2] = torch.sin(pos * div_term) # Every second column starting from index 0
        pe[:, 1::2] = torch.cos(pos * div_term) # For all odd indices cosinusoidal values

        # New batch dimension: (1, max_patches, emb_dim)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :] # (batch_size, seq_len, d_model)
        return x

In [ ]:
# Transformer components

class ResidualConnection(nn.Module):
    """
    Residual connection for faster and more efficient Transformer training
    
    Parameters:
    - block: Network the residual connection is applied to (e. g. feed forward block)
    - emb_dim: Embedding dimension
    - dropout_rate: Dropout rate
    """

    def __init__(self, block, emb_dim, dropout_rate=0.1):
        super(ResidualConnection, self).__init__()
        self.block = block
        self.norm = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.norm(x + self.block(x))
        x = self.dropout(x)
        return x


class FeedForwardBlock(nn.Module):
    """
    Feed-forward block of the Transformer, a simple MLP
    
    Parameters:
    - in_dim: Dimension of the input
    - exp_fct: Expansion factor, how much to expand hidden layer
    - dropout_rate: Dropout rate
    """

    def __init__(self, in_dim, exp_fct=4, dropout_rate=0.1):
        super(FeedForwardBlock, self).__init__()

        hidden_dim = in_dim * exp_fct

        self.linear_in = nn.Linear(in_dim, hidden_dim)
        self.silu = nn.SiLU()
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.linear_out = nn.Linear(hidden_dim, in_dim)

    def forward(self, x):
        x = self.linear_in(x)
        x = self.relu(x) # Switch to SiLU to experiment
        x = self.dropout(x)
        x = self.linear_out(x)
        return x

Multi-head attention returns a tuple of output and weights, but the residual connection class expects a single tensor. Therefore, a wrapper (class AttentionWrapper) is constructed to ensure that the residual connections receive input of correct dimension. The EncoderBlock class then combines all Transformer elements into the Encoder-only architecture.

In [ ]:
class AttentionWrapper(nn.Module):
    """
    Helper to wrap MultiheadAttention because MHA returns tuples
    but ResidualConnection expects a single tensor.
    """
    def __init__(self, emb_dim, n_heads, dropout_rate):
        super(AttentionWrapper, self).__init__()
        self.mha = nn.MultiheadAttention(emb_dim, n_heads, dropout_rate, batch_first=True)

    def forward(self, x):
        # Only return the output tensor, not the attention weights
        out, _ = self.mha(x, x, x)
        return out

class EncoderBlock(nn.Module):
    """
    Encoder-Only Transformer to capture global relationships using multihead attention
    
    Parameters:
    - emb_dim: Embedding dimension
    - n_heads: Number of heads for MHA
    - dropout_rate: Dropout rate
    - expansion: Expansion rate
    """

    def __init__(self, emb_dim, n_heads, dropout_rate=0.1, expansion=4):
        super(EncoderBlock, self).__init__()

        self.attention = AttentionWrapper(emb_dim, n_heads, dropout_rate)
        self.ffn = FeedForwardBlock(emb_dim, expansion, dropout_rate)
        self.residual1 = ResidualConnection(self.attention, emb_dim, dropout_rate)
        self.residual2 = ResidualConnection(self.ffn, emb_dim, dropout_rate)

    def forward(self, x, mask=None):
        x = self.residual1(x)
        x = self.residual2(x)
        return x

In [ ]:
class MLPClassifier(nn.Module):
    """
    Final MLP Classifier Layer
    
    Parameters:
    - tr_out_dim: Dimension of transformer output
    """

    def __init__(self, tr_out_dim, dropout_rate=0.1):
        super(MLPClassifier, self).__init__()

        self.linear_in = nn.Linear(tr_out_dim * 2, tr_out_dim // 2)
        self.linear_out = nn.Linear(tr_out_dim // 2, 1)
        self.relu = nn.ReLU(True)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.linear_in(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear_out(x)
        return x

In [ ]:
# Final model

class EEGClassifier(nn.Module):
    """
    Main CNN-Transformer Model

    Parameters:
    - in_dim: Dimension of input, corresponding to number of input channels (64)
    - emb_dim: Dimension of embeddings
    - max_patches: Maximum of patches for positional encoding
    - n_channels: Number of EEG channels
    - n_heads: Number of heads for multihead attention
    - dropout_rate: Dropout rate
    - tr_out_dim: Transformer output dimension
    """

    def __init__(self, in_dim=64, emb_dim=64, max_patches=, 
                 n_channels=64, n_heads=4, dropout_rate=0.5, tr_out_dim=):
        super(EEGClassifier, self).__init__()
    
        self.positional_encoding = PositionalEncoding(emb_dim, max_patches) 
        self.cnn = CNN()
        self.transformer = EncoderBlock(in_dim, emb_dim, n_heads, dropout_rate)
        self.mlp = MLPClassifier(in_dim) # Assuming pooling reduces patches by half

    def forward(self, x):
        """
        Args: x: Input EEG data (batch_size, n_channels, num_time_points)
        Returns: logits for classification
        """
        x = self.cnn(x)             # (batch_size, n_channels, n_samples)
        x = self.positional_encoding(x)
        x = x.permute(2, 0, 1)   # (n_samples, batch_size, n_channels)
        x = self.transformer(x)     # (n_samples, batch_size, n_channels)
        x = self.mlp(x)
        
        return x.squeeze() # Squeeze to remove single dim for binary classification

### Train.py
_Loading and preprocessing the data, passing it through the baseline MLP and then main model, optimizing via the Adam optimizer and training both the baseline and main model._

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuddn.deterministic = True

In [ ]:
# Function to train model
def train_model(model, x, num_epochs, train_loader, optimizer):
    # Setup
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.train()

    for epoch in range(num_epochs):
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            # Forward pass
            optimizer.zero_grad()
            pred = model(batch_X)
            loss = nn.CrossEntropyLoss(pred, batch_y)

            # Backward pass
            loss.backward()
            optimizer.step()

        print(loss.item)

In [ ]:
# Load data
dataset = PreprocessedDataset(subject_ids=[1], runs=[4])
X, y = dataset.load_data()

sample, label = dataset[0]
print(f"Sample shape: {sample.shape}, Label: {label}")

In [ ]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

# Create train and test dataloaders
train_loader, test_loader = create_dataloaders(X, y)

# Training loop for main model
num_epochs = 8 # number of epochs

#### Hyperparameter Tuning of the baseline MLP
1. Size of hidden dimension:
* hidden_dim = in_dim - test if a linear classifier like Logistic Regression would be sufficient
* hidden_dim < in_dim (compression) - assumption of highly correlated spectral features due to simple task 
* hidden_dim > in_dim (expansion) - assume a highly complex relationship between features and output class (increases risk of overfitting)

-> compression seems the most intuitive solution to me since the underlying task was not too difficult

In [ ]:
n_channels = 64
n_samples = 4.5 * 160 # time_window * sampling_freq
in_dim = n_channels * n_samples
hidden_dim = 256

model_baseline = PoorMLP(in_dim, hidden_dim)

In [ ]:
# hidden_dim and dropout could also be modified if desired (and then plotted)
optimizer_baseline = optim.Adam(model_baseline.parameters(), lr=0.001)
train_model(model_baseline, X, num_epochs, train_loader, optimizer_baseline)

"""
If the train_model fct does not work:

for epoch in range(num_epochs):
    for batch in dataloader:
        train(model_baseline, train_loader)
        optimizer_baseline.zero_grad()
        pred = model_baseline(x)
        loss = nn.CrossEntropyLoss(pred, x)
        loss.backward()
        optimizer_baseline.step()
    print(loss.item)"""

#### Training the main CNN-Transformer

In [ ]:
# model_main = EEGClassifier()

optimizer_main = optim.Adam(model_main.parameters(), lr=0.001)
# train_model(model_main, X, num_epochs, Prepro, train_loader, optimizer_main)

### Evaluate.py
_compare the baseline MLP vs. CNN-Transformer performance using accuracy and plotting other performance metrics_

In [ ]:
def accuracy(model, test_loader):
    model.load_state_dict(torch.load("checkpoint.pt"))
    model.eval()

    correct = 0

    with torch.no_grad():
        for x, y in test_loader:
            output = model(x)
            pred = output.argmax(dim=1)
            correct += (pred==y).sum()

    acc = correct / len(test_loader.dataset)
    return {'acc': acc} 
    # Change dictionary item output perhaps

In [ ]:
metrics = {
    'Baseline Linear': accuracy(model_baseline_linear, test_loader),
    'Baseline Compression': accuracy(model_baseline_compr, test_loader),
    'Baseline Expansion': accuracy(model_baseline_expansion, test_loader)
}

fig, ax = plt.subplots(figsize=(13, 5))
for model, metric in metrics.items():
    ax.plot([model], [metric['acc']], marker='o', label=model)

ax.set_xlabel('Model')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Model Performance Comparison')
ax.legend()
plt.tight_layout()
plt.show()

"""
If accuracy fct does not work:


model_baseline.load_state_dict(torch.load("checkpoint.pt"))
model_baseline.eval()

correct = 0

with torch.no_grad():
    for x, y in test_loader:
        output_baseline = model_baseline(x)
        pred_baseline = output_baseline.argmax(dim=1)
        correct_baseline += (pred_baseline==y).sum()

acc_baseline = correct_baseline / len(test_loader.dataset)
print(f"Accuracy of the baseline MLP: {acc_baseline}")"""

In [ ]:
"""model_main = EEGClassifier() # add params here
accuracy(model_main, test_loader)"""